# Урок 15. K-Means и PCA — обучение без учителя
**Библиотеки:** numpy, sklearn, matplotlib

**Цель:** находить группы в данных БЕЗ ответов и сжимать размерность.

## Простыми словами
Unsupervised: есть только X, ответов y нет. Задача — найти структуру.

**K-Means** ищет K групп (кластеров):
1. Кинуть K случайных центров (центроидов).
2. Каждую точку приписать к ближайшему центру.
3. Передвинуть каждый центр в середину его точек.
4. Повторять 2–3, пока центры не остановятся.

**Как выбрать K?** Elbow-метод: график inertia (сумма расстояний точек до своих центров)
для разных K. Ищем «локоть» — место, где улучшение резко замедляется.

**PCA** — сжатие размерности: из 10 признаков сделать 2 «суперпризнака», сохранив максимум
информации. Главное применение для нас — нарисовать многомерные данные на плоскости.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Клиенты магазина: доход и трат — БЕЗ ответов
rng = np.random.default_rng(10)
g1 = rng.normal([30, 20], 6, (70, 2))    # экономные
g2 = rng.normal([70, 80], 8, (70, 2))    # богатые транжиры
g3 = rng.normal([75, 25], 7, (60, 2))    # богатые экономные
X = np.vstack([g1, g2, g3])
X_s = StandardScaler().fit_transform(X)   # kmeans меряет расстояния -> scaling обязателен

km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X_s)
labels = km.labels_                        # номер кластера каждой точки

plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=20)
centers = StandardScaler().fit(X).inverse_transform(km.cluster_centers_)
plt.scatter(centers[:, 0], centers[:, 1], c='red', s=250, marker='X', label='центроиды')
plt.xlabel('доход'); plt.ylabel('траты'); plt.legend(); plt.grid(True); plt.show()

In [ ]:
# Elbow-метод для выбора K
inertias = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_s).inertia_ for k in range(1, 9)]
plt.plot(range(1, 9), inertias, 'o-')
plt.xlabel('K'); plt.ylabel('inertia'); plt.title('Локоть на K=3'); plt.grid(True); plt.show()

In [ ]:
# PCA: жмём 64-мерные цифры в 2D и смотрим
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits
d = load_digits()
pts = PCA(n_components=2).fit_transform(d.data)    # 64 признака -> 2
plt.figure(figsize=(8, 6))
sc = plt.scatter(pts[:, 0], pts[:, 1], c=d.target, cmap='tab10', s=8)
plt.colorbar(sc, label='цифра'); plt.title('1797 цифр (64D) на плоскости через PCA'); plt.show()

## Что произошло
K-Means без всяких ответов нашёл 3 группы клиентов — маркетолог теперь может работать с каждой
по-разному. Elbow подтвердил K=3. PCA сжал 64-пиксельные цифры до 2 чисел — и одинаковые
цифры легли рядом: данные имеют структуру, и её видно глазами.

## Типичные ошибки
1. KMeans без scaling — признак с большими числами захватит все расстояния.
2. Ждать от KMeans «правильных» групп — он находит геометрические сгустки, смысл им даёшь ты.
3. Забыть n_init: результат зависит от случайного старта; n_init=10 запускает 10 раз и берёт лучший.

## Конспект 📓
Unsupervised: только X. KMeans: центры → приписка → сдвиг → повтор. labels_ — кластеры,
cluster_centers_ — центры, inertia_ — компактность. Выбор K — elbow. Scaling обязателен.
PCA(n_components=2) — сжать и нарисовать многомерное.

---
## Мини-задание 💪
Добавь четвёртую группу клиентов (низкий доход, высокие траты — «кредитные»). Перестрой elbow: сместился ли локоть на 4?

## 5 проверочных вопросов ❓
1. Чем unsupervised отличается от supervised?
2. Опиши 4 шага K-Means.
3. Что такое inertia и как elbow помогает выбрать K?
4. Зачем scaling перед KMeans?
5. Что делает PCA и зачем он нам чаще всего?

## Домашнее задание 🏠
Задачи 33–34, 43, 49 из practice_tasks.md.
